## 华泰量价行业择时策略

定义了两个行业择时因子，当两个因子值均上升，且累计成交量同比因子大于0时给予行业看多信号，当看多行业大于等于5个时配置该行业，所有待配置的行业保持相等权重

In [1]:
# =====================================================
# 华泰价量择时模型：策略三（申万一级行业指数）
#
# 使用方式：修改“策略参数区”中的日期等参数后，整段在 BigQuant
# Notebook 中直接运行。因子层仍只计算因子；本文件负责策略三的
# 信号、调仓日映射与真实 A 股代理组合的 BigTrader 回测。
#
# 时序：月末 T 日收盘后计算信号，严格在 T+1 交易日开盘调仓。
# =====================================================

import hashlib
from datetime import timedelta

import numpy as np
import pandas as pd

import dai
from bigquant import bigtrader

from factor_lib.common.data_adapters.bigquant_adapters.industry_daily import (
    load_industry_daily_raw_data,
)
from factor_lib.common.data_adapters.bigquant_adapters.loader import (
    load_factor_raw_data,
)
from factor_lib.factor_hub.get_factor import get_factor


# =====================================================
# 1. 策略参数区：通常只需要修改这里
# =====================================================

# 回测区间。策略三需要至少约两年的预热数据，脚本会自动向前取数。
START_DATE = "2014-01-01"
END_DATE = "2026-08-31"

# 本策略使用当前申万 2021 一级行业体系；行业指数代码在运行时从
# cn_stock_factors_company_profile 读取，而不是写死研报年代的 28 个代码。
INDUSTRY_INDICES = None

# 策略三：当月满足三项条件的行业数不少于该值时开仓，否则全额空仓。
MIN_QUALIFIED_INDUSTRIES = 5

# 开仓时行业间等权；每个行业内按可交易股票的流通市值加权。
TARGET_TOTAL_WEIGHT = 0.98
CAPITAL_BASE = 1_000_000

# 真实股票代理组合的股票池和交易参数。默认覆盖四类已知 A 股上市板块：
# 主板、创业板、科创板和北交所；不纳入板块代码未知的证券。
ALLOWED_LIST_SECTORS = [1, 2, 3, 4]
MIN_LIST_DAYS = 60
# None 表示保留行业内全部合格股票；设为正整数时，只保留流通市值最大的 N 只。
MAX_STOCKS_PER_INDUSTRY = None
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COST = 5.0
VOLUME_LIMIT = 0.025

# 数据加载、因子计算与回测调仓阶段的简洁进度信息。
SHOW_PROGRESS = True
PROGRESS_EVERY = 12


# =====================================================
# 2. 一次性准备行业因子、月度信号和 T+1 调仓表
# =====================================================

if pd.Timestamp(START_DATE) > pd.Timestamp(END_DATE):
    raise ValueError("START_DATE 不能晚于 END_DATE。")
if MIN_QUALIFIED_INDUSTRIES < 1:
    raise ValueError("MIN_QUALIFIED_INDUSTRIES 必须为正整数。")
if not 0 < TARGET_TOTAL_WEIGHT <= 1:
    raise ValueError("TARGET_TOTAL_WEIGHT 必须位于 (0, 1]。")
if MAX_STOCKS_PER_INDUSTRY is not None and (
    not isinstance(MAX_STOCKS_PER_INDUSTRY, int)
    or isinstance(MAX_STOCKS_PER_INDUSTRY, bool)
    or MAX_STOCKS_PER_INDUSTRY <= 0
):
    raise ValueError("MAX_STOCKS_PER_INDUSTRY 必须为正整数或 None。")

# 以回测结束日前最近 90 个自然日的资料表确定本次使用的申万 2021 一级
# 行业指数集合。该集合是当前分类体系的静态研究口径，并非逐期未来筛选。
mapping_start_date = (
    pd.Timestamp(END_DATE) - timedelta(days=90)
).strftime("%Y-%m-%d")
industry_mapping = dai.query(
    """
    SELECT DISTINCT sw_level_index_code AS instrument
    FROM cn_stock_factors_company_profile
    WHERE sw_level_index_code IS NOT NULL
    ORDER BY instrument
    """,
    filters={"date": [mapping_start_date, END_DATE]},
).df()
INDUSTRY_INDICES = sorted(industry_mapping["instrument"].dropna().astype(str).unique())
if not INDUSTRY_INDICES:
    raise ValueError("未查询到申万 2021 一级行业指数代码。")
if SHOW_PROGRESS:
    print(f"[策略三] 当前申万一级行业指数：{len(INDUSTRY_INDICES)} 个。")

# 成交量因子需覆盖连续 24 个自然月；840 个自然日为查询缓冲，不改变公式。
RAW_START_DATE = (
    pd.Timestamp(START_DATE) - timedelta(days=840)
).strftime("%Y-%m-%d")

FACTOR_PARAMS = {"industry_indices": INDUSTRY_INDICES}

if SHOW_PROGRESS:
    print("[策略三] [1/4] 加载行业价格同比因子所需原始数据...")
price_raw_bundle = load_factor_raw_data(
    factor_name="industry_price_log_yoy",
    start_date=RAW_START_DATE,
    end_date=END_DATE,
    factor_params=FACTOR_PARAMS,
    show_progress=SHOW_PROGRESS,
)
price_factor = get_factor(
    "industry_price_log_yoy",
    price_raw_bundle,
    as_of_date=END_DATE,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    **FACTOR_PARAMS,
)

if SHOW_PROGRESS:
    print("[策略三] [2/4] 加载行业累计成交量同比因子所需原始数据...")
volume_raw_bundle = load_factor_raw_data(
    factor_name="industry_cumulative_volume_log_yoy",
    start_date=RAW_START_DATE,
    end_date=END_DATE,
    factor_params=FACTOR_PARAMS,
    show_progress=SHOW_PROGRESS,
)
volume_factor = get_factor(
    "industry_cumulative_volume_log_yoy",
    volume_raw_bundle,
    as_of_date=END_DATE,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    **FACTOR_PARAMS,
)

# 用相同行业指数行情建立月末信号日和下一交易日开盘的执行日。
if SHOW_PROGRESS:
    print("[策略三] [3/4] 加载行业指数执行行情并构造月度调仓表...")
industry_bars = load_industry_daily_raw_data(
    standard_fields=[
        "industry_open",
        "industry_high",
        "industry_low",
        "industry_close",
        "industry_volume",
    ],
    industry_indices=INDUSTRY_INDICES,
    start_date=START_DATE,
    end_date=END_DATE,
    show_progress=SHOW_PROGRESS,
)

if industry_bars.empty:
    raise ValueError("未获取到行业指数行情，请检查 INDUSTRY_INDICES 与日期区间。")

price_factor = price_factor.rename(
    columns={"industry_price_log_yoy": "price_log_yoy"}
)
volume_factor = volume_factor.rename(
    columns={"industry_cumulative_volume_log_yoy": "volume_log_yoy"}
)

factor_panel = price_factor.merge(
    volume_factor,
    on=["date", "instrument"],
    how="inner",
    validate="one_to_one",
)
factor_panel["date"] = pd.to_datetime(factor_panel["date"]).dt.normalize()
factor_panel = factor_panel.sort_values(
    ["instrument", "date"], kind="mergesort"
).reset_index(drop=True)

# 研报策略三：PL_t > PL_{t-1}，VL_t > VL_{t-1}，且 VL_t > 0。
# 这里不额外要求 PL_t > 0。
factor_panel["previous_date"] = factor_panel.groupby("instrument")["date"].shift(1)
factor_panel["previous_price_log_yoy"] = factor_panel.groupby("instrument")[
    "price_log_yoy"
].shift(1)
factor_panel["previous_volume_log_yoy"] = factor_panel.groupby("instrument")[
    "volume_log_yoy"
].shift(1)
factor_panel["is_consecutive_month"] = (
    factor_panel["date"].dt.to_period("M")
    == factor_panel["previous_date"].dt.to_period("M") + 1
)
factor_panel["qualified"] = (
    factor_panel["is_consecutive_month"]
    & (factor_panel["price_log_yoy"] > factor_panel["previous_price_log_yoy"])
    & (factor_panel["volume_log_yoy"] > factor_panel["previous_volume_log_yoy"])
    & (factor_panel["volume_log_yoy"] > 0)
)

# 每个自然月最后一个行业交易日是信号日；即使没有行业满足条件也保留该日期，
# 这样阈值不足时可以在下一交易日明确清仓。
trading_dates = pd.DatetimeIndex(
    pd.to_datetime(industry_bars["date"]).dt.normalize().unique()
).sort_values()
monthly_signal_dates = pd.DataFrame({"signal_date": trading_dates})
monthly_signal_dates["month"] = monthly_signal_dates["signal_date"].dt.to_period("M")
monthly_signal_dates = (
    monthly_signal_dates.groupby("month", as_index=False)["signal_date"]
    .max()
    .drop(columns="month")
)
qualified_counts = (
    factor_panel.loc[factor_panel["qualified"]]
    .groupby("date", as_index=False)["instrument"]
    .nunique()
    .rename(columns={"date": "signal_date", "instrument": "qualified_count"})
)
rebalance_schedule = monthly_signal_dates.merge(
    qualified_counts,
    on="signal_date",
    how="left",
).fillna({"qualified_count": 0})
rebalance_schedule["qualified_count"] = rebalance_schedule["qualified_count"].astype(int)
rebalance_schedule["risk_on"] = (
    rebalance_schedule["qualified_count"] >= MIN_QUALIFIED_INDUSTRIES
)

# 信号 T 日已经使用了 T 日收盘数据，必须匹配到严格晚于 T 的首个交易日。
execution_calendar = pd.DataFrame({"execution_date": trading_dates})
rebalance_schedule = pd.merge_asof(
    rebalance_schedule.sort_values("signal_date"),
    execution_calendar.sort_values("execution_date"),
    left_on="signal_date",
    right_on="execution_date",
    direction="forward",
    allow_exact_matches=False,
)
rebalance_schedule = rebalance_schedule.dropna(subset=["execution_date"])
rebalance_schedule = rebalance_schedule.loc[
    rebalance_schedule["execution_date"] <= pd.Timestamp(END_DATE)
].copy()

if rebalance_schedule.empty:
    raise ValueError("回测区间内没有可执行的 T+1 调仓日。")

qualified_industries = factor_panel.loc[
    factor_panel["qualified"], ["date", "instrument"]
].rename(columns={"date": "signal_date"})
qualified_industries = qualified_industries.merge(
    rebalance_schedule.loc[
        rebalance_schedule["risk_on"],
        ["signal_date", "execution_date", "qualified_count"],
    ],
    on="signal_date",
    how="inner",
    validate="many_to_one",
)
qualified_industries["industry_weight"] = (
    TARGET_TOTAL_WEIGHT / qualified_industries["qualified_count"]
)
qualified_industries = qualified_industries.sort_values(
    ["execution_date", "instrument"], kind="mergesort"
).reset_index(drop=True)
if qualified_industries.empty:
    raise ValueError(
        "回测区间内没有任何月份满足策略三的开仓阈值；"
        "请检查日期区间或 MIN_QUALIFIED_INDUSTRIES。"
    )

# =====================================================
# 3. 将行业配置映射为真实股票目标权重
# =====================================================

if SHOW_PROGRESS:
    print("[策略三] [4/5] 读取信号日的申万行业成分和股票状态...")
signal_dates_for_stock = sorted(
    qualified_industries["signal_date"].dt.strftime("%Y-%m-%d").unique()
)
signal_dates_sql = ", ".join(f"'{date}'" for date in signal_dates_for_stock)
industry_indices_sql = ", ".join(f"'{code}'" for code in INDUSTRY_INDICES)
list_sectors_sql = ", ".join(str(int(value)) for value in ALLOWED_LIST_SECTORS)

stock_candidates = dai.query(
    f"""
    SELECT
        profile.date,
        profile.instrument,
        profile.sw_level_index_code AS industry_index,
        valuation.float_market_cap
    FROM cn_stock_factors_company_profile AS profile
    INNER JOIN cn_stock_prefactors AS prefactor
        ON profile.date = prefactor.date
        AND profile.instrument = prefactor.instrument
    INNER JOIN cn_stock_valuation AS valuation
        ON profile.date = valuation.date
        AND profile.instrument = valuation.instrument
    WHERE profile.date IN ({signal_dates_sql})
      AND profile.sw_level_index_code IN ({industry_indices_sql})
      AND prefactor.list_sector IN ({list_sectors_sql})
      AND prefactor.is_risk_warning = 0
      AND prefactor.suspended = 0
      AND prefactor.list_days >= {MIN_LIST_DAYS}
      AND valuation.float_market_cap > 0
    ORDER BY profile.date, profile.sw_level_index_code, valuation.float_market_cap DESC,
             profile.instrument
    """,
    filters={
        "date": [
            min(signal_dates_for_stock),
            max(signal_dates_for_stock),
        ]
    },
).df()

stock_candidates["date"] = pd.to_datetime(stock_candidates["date"]).dt.normalize()
if stock_candidates.duplicated(["date", "instrument"], keep=False).any():
    raise ValueError("股票行业成分查询在 date + instrument 上出现重复记录。")

stock_targets = qualified_industries.merge(
    stock_candidates,
    left_on=["signal_date", "instrument"],
    right_on=["date", "industry_index"],
    how="inner",
    validate="one_to_many",
).drop(columns=["date"])

# 合并后的 instrument_x 与 industry_index 都表示同一行业代码：前者来自
# 因子信号，后者来自股票映射。保留两列会在后续改名时产生同名列，进而导致
# pandas 无法按行业分组；先校验一致性，再只保留一份行业键。
if not stock_targets["instrument_x"].eq(stock_targets["industry_index"]).all():
    raise ValueError("行业信号代码与信号日股票行业映射不一致。")
stock_targets = stock_targets.drop(columns=["industry_index"])

if MAX_STOCKS_PER_INDUSTRY is not None:
    stock_targets = (
        stock_targets.sort_values(
            ["signal_date", "instrument_x", "float_market_cap", "instrument_y"],
            ascending=[True, True, False, True],
            kind="mergesort",
        )
        .groupby(["signal_date", "instrument_x"], group_keys=False)
        .head(MAX_STOCKS_PER_INDUSTRY)
        .reset_index(drop=True)
    )

stock_targets = stock_targets.rename(
    columns={"instrument_x": "industry_index", "instrument_y": "instrument"}
)
stock_targets["industry_float_market_cap"] = stock_targets.groupby(
    ["signal_date", "industry_index"]
)["float_market_cap"].transform("sum")
stock_targets["weight"] = (
    stock_targets["industry_weight"]
    * stock_targets["float_market_cap"]
    / stock_targets["industry_float_market_cap"]
)
stock_targets = stock_targets.sort_values(
    ["execution_date", "weight", "instrument"],
    ascending=[True, False, True],
    kind="mergesort",
).reset_index(drop=True)
stock_targets["target_rank"] = stock_targets.groupby("execution_date").cumcount() + 1
stock_targets["is_cash_signal"] = False

# 风险关闭月、或入选行业在股票池中没有任何可交易成分时，写入现金标记，
# 使 BigTrader 在该执行日主动清仓。标记不是交易标的。
target_dates = set(pd.to_datetime(stock_targets["execution_date"]).dt.normalize())
cash_signals = rebalance_schedule.loc[
    ~rebalance_schedule["execution_date"].isin(target_dates),
    ["signal_date", "execution_date", "qualified_count", "risk_on"],
].copy()
cash_signals["instrument"] = "__CASH_SIGNAL__"
cash_signals["industry_index"] = ""
cash_signals["weight"] = 0.0
cash_signals["target_rank"] = 0
cash_signals["is_cash_signal"] = True

signal_df = pd.concat(
    [
        stock_targets[
            [
                "execution_date", "signal_date", "instrument", "industry_index",
                "weight", "target_rank", "qualified_count", "is_cash_signal",
            ]
        ],
        cash_signals[
            [
                "execution_date", "signal_date", "instrument", "industry_index",
                "weight", "target_rank", "qualified_count", "is_cash_signal",
            ]
        ],
    ],
    ignore_index=True,
).rename(columns={"execution_date": "date"})
signal_df["date"] = pd.to_datetime(signal_df["date"]).dt.strftime("%Y-%m-%d")
signal_df["signal_date"] = pd.to_datetime(signal_df["signal_date"]).dt.strftime("%Y-%m-%d")
signal_df = signal_df.sort_values(
    ["date", "is_cash_signal", "target_rank", "instrument"], kind="mergesort"
).reset_index(drop=True)

if signal_df.empty:
    raise ValueError("未形成任何真实股票调仓信号。")

check_df = signal_df[
    ["date", "signal_date", "industry_index", "instrument", "weight", "qualified_count"]
].copy()
signal_hash = hashlib.md5(check_df.to_csv(index=False).encode("utf-8")).hexdigest()
engine_instruments = sorted(
    signal_df.loc[~signal_df["is_cash_signal"], "instrument"].unique()
)
if not engine_instruments:
    raise ValueError("所有开仓月份均未找到可交易股票，无法启动 BigTrader。")
target_weight_by_date = stock_targets.groupby("execution_date")["weight"].sum()
underinvested_count = int((target_weight_by_date < TARGET_TOTAL_WEIGHT - 1e-8).sum())

print(
    f"[策略三] [5/5] 股票信号准备完成：{signal_df['date'].nunique()} 次调仓，"
    f"开仓 {stock_targets['execution_date'].nunique() if not stock_targets.empty else 0} 次，"
    f"涉及 {len(engine_instruments):,} 只股票，信号哈希 {signal_hash}。"
)
if underinvested_count:
    print(
        "[策略三] 提示："
        f"{underinvested_count} 个开仓月存在入选行业无合格股票，"
        "对应行业资金将保留为现金。"
    )


# =====================================================
# 4. BigTrader 原生股票回测
# =====================================================

def initialize(context: bigtrader.IContext):
    context.signal_data = context.data.copy()
    context.signal_data["date"] = pd.to_datetime(
        context.signal_data["date"]
    ).dt.strftime("%Y-%m-%d")
    context.signal_dates = set(context.signal_data["date"].unique())
    context.completed_rebalances = 0
    context.total_rebalances = len(context.signal_dates)
    context.desired_weight_map = {}
    context.set_commission(
        bigtrader.PerOrder(
            buy_cost=BUY_COST,
            sell_cost=SELL_COST,
            min_cost=MIN_COST,
        )
    )
    try:
        from bigtrader.constant import VMatchAt

        context.set_vmatch_at(VMatchAt.CURRENT_BAR)
    except Exception:
        pass


def handle_data(context: bigtrader.IContext, data: bigtrader.IBarData):
    today = data.current_dt.strftime("%Y-%m-%d")
    if today not in context.signal_dates:
        return

    today_signal = context.signal_data.loc[
        context.signal_data["date"] == today
    ].copy()
    today_targets = today_signal.loc[
        ~today_signal["is_cash_signal"]
    ].sort_values(["target_rank", "instrument"], kind="mergesort")
    target_instruments = set(today_targets["instrument"])

    # 卖出先于买入；开盘跌停、停牌或无成交量时保留持仓，不能假定卖出成功。
    for instrument in sorted(set(context.get_positions().keys()) - target_instruments):
        try:
            bar = data.current(instrument, ["open", "lower_limit", "volume"])
            can_sell = (
                float(bar["volume"]) > 0
                and float(bar["open"]) > float(bar["lower_limit"])
            )
        except Exception:
            can_sell = False
        if can_sell:
            submit_result = context.order_target_percent(instrument, 0)
            try:
                submitted = int(submit_result) >= 0
            except (TypeError, ValueError):
                submitted = submit_result is None
            if submitted:
                context.desired_weight_map[instrument] = 0.0

    # 目标权重上调按买入约束、下调按卖出约束处理，不能将减仓误判为买入。
    for row in today_targets.itertuples(index=False):
        target_weight = float(row.weight)
        previous_weight = float(context.desired_weight_map.get(row.instrument, 0.0))
        weight_change = target_weight - previous_weight
        if abs(weight_change) < 1e-8:
            continue
        try:
            bar = data.current(
                row.instrument,
                ["open", "upper_limit", "lower_limit", "volume"],
            )
            can_trade = float(bar["volume"]) > 0
            if weight_change > 0:
                can_trade = can_trade and (
                    float(bar["open"]) < float(bar["upper_limit"])
                )
            else:
                can_trade = can_trade and (
                    float(bar["open"]) > float(bar["lower_limit"])
                )
        except Exception:
            can_trade = False
        if can_trade:
            submit_result = context.order_target_percent(row.instrument, target_weight)
            try:
                submitted = int(submit_result) >= 0
            except (TypeError, ValueError):
                submitted = submit_result is None
            if submitted:
                context.desired_weight_map[row.instrument] = target_weight

    context.completed_rebalances += 1
    if SHOW_PROGRESS and (
        context.completed_rebalances == 1
        or context.completed_rebalances % PROGRESS_EVERY == 0
        or context.completed_rebalances == context.total_rebalances
    ):
        first_row = today_signal.iloc[0]
        print(
            "[策略三股票回测] "
            f"调仓 {context.completed_rebalances}/{context.total_rebalances} | "
            f"执行日 {today} | 信号日 {first_row['signal_date']} | "
            f"满足条件 {int(first_row['qualified_count'])} 个 | "
            f"目标股票 {len(today_targets)} 只"
        )


performance = bigtrader.run(
    market=bigtrader.Market.CN_STOCK,
    frequency=bigtrader.Frequency.DAILY,
    start_date=signal_df["date"].min(),
    end_date=END_DATE,
    capital_base=CAPITAL_BASE,
    instruments=engine_instruments,
    data=signal_df,
    initialize=initialize,
    handle_data=handle_data,
    benchmark="000300.SH",
    order_price_field_buy="open",
    order_price_field_sell="open",
    volume_limit=VOLUME_LIMIT,
)


[策略三] 当前申万一级行业指数：31 个。
[策略三] [1/4] 加载行业价格同比因子所需原始数据...
[BigQuant 行业日频适配器] [3/3] 行业日频数据准备完成 | 112,623 行 | 已耗时 0.1s                                                                                                                         
[BigQuant loader] 1/1（100.00%），数据域合并完成，当前 industry_daily，112,623 行，耗时 0.1s                                                                                                         
[industry_price_log_yoy] 已完成行业指数计算 | 31/31 个行业指数（100.0%） | 当前：801980.SWI | 已耗时：0.2s                                                                                                
[策略三] [2/4] 加载行业累计成交量同比因子所需原始数据...
[BigQuant 行业日频适配器] [3/3] 行业日频数据准备完成 | 112,623 行 | 已耗时 0.1s                                                                                                                         
[BigQuant loader] 1/1（100.00%），数据域合并完成，当前 industry_daily，112,623 行，耗时 0.1s                                                                                                         
[industry_

[2026-09-18 12:19:48] [info     ] bigtrader run done.
